# 04 - Module API 方法

## 学习目标

1. 掌握 `train()` / `eval()` 模式切换
2. 学会使用 `to(device)` / `cpu()` / `cuda()` 进行设备转移
3. 掌握 `state_dict()` 和 `load_state_dict()` 进行模型保存与加载
4. 理解 `parameters()` / `named_parameters()` / `modules()` / `children()` 的区别
5. 了解精度转换方法和 `apply()` 的用法

---

In [ ]:
import torch
import torch.nn as nn
import os
import tempfile

In [ ]:
# 定义一个示例模型
class DemoNet(nn.Module):
    """用于演示 Module API 的网络。"""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 28 * 28, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = DemoNet()
print(model)

## 1. train() vs eval() 模式切换

**核心区别：** 影响 `BatchNorm` 和 `Dropout` 的行为。

| 模式 | BatchNorm | Dropout |
|------|-----------|----------|
| `train()` | 用当前 batch 统计量，更新 running stats | 随机置零 |
| `eval()` | 用 running stats，不更新 | 不做操作 |

In [ ]:
# train() 和 eval() 切换
print(f"默认 training 状态: {model.training}")

# 切换到评估模式
model.eval()
print(f"eval() 后: {model.training}")

# 切换回训练模式
model.train()
print(f"train() 后: {model.training}")

# 所有子模块也会同步切换
model.eval()
print(f"\n子模块状态:")
for name, module in model.named_modules():
    if hasattr(module, 'training') and name:
        print(f"  {name}: training={module.training}")

In [ ]:
# 验证 eval 模式对输出的影响
x = torch.randn(4, 1, 28, 28)

model.train()
out_train1 = model(x)
out_train2 = model(x)
print(f"训练模式: 两次输出相同吗？{torch.equal(out_train1, out_train2)}")
print("（因为 Dropout 是随机的，所以训练模式下两次结果不同）")

model.eval()
with torch.no_grad():
    out_eval1 = model(x)
    out_eval2 = model(x)
print(f"\n评估模式: 两次输出相同吗？{torch.equal(out_eval1, out_eval2)}")
print("（Dropout 关闭，BN 用固定统计量，所以结果一致）")

**最佳实践：**

```python
# 训练时
model.train()
output = model(x)
loss.backward()

# 评估/推理时
model.eval()
with torch.no_grad():
    output = model(x)
```

## 2. 设备转移：to(), cpu(), cuda()

将模型和数据移到同一个设备上（CPU 或 GPU）。

In [ ]:
model = DemoNet()

# 查看参数所在设备
print(f"默认设备: {next(model.parameters()).device}")

# 使用 to() 转移设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"to({device}) 后: {next(model.parameters()).device}")

# cpu() 和 cuda()
model_cpu = model.cpu()
print(f"cpu() 后: {next(model_cpu.parameters()).device}")

if torch.cuda.is_available():
    model_gpu = model.cuda()
    print(f"cuda() 后: {next(model_gpu.parameters()).device}")

In [ ]:
# 注意：数据也要移到同一设备
model = DemoNet().to(device)
x = torch.randn(2, 1, 28, 28).to(device)  # 数据也要 to(device)

model.eval()
with torch.no_grad():
    output = model(x)
print(f"模型设备: {next(model.parameters()).device}")
print(f"数据设备: {x.device}")
print(f"输出设备: {output.device}")

## 3. state_dict() 与 load_state_dict()

模型保存与加载的核心方法。

- `state_dict()`：返回模型所有参数和缓冲区的字典
- `load_state_dict()`：从字典加载参数

In [ ]:
model = DemoNet()

# 查看 state_dict
print("state_dict 的 keys:")
for key, value in model.state_dict().items():
    print(f"  {key:40s}  shape={list(value.shape)}")

In [ ]:
# 保存与加载模型（推荐方式：只保存 state_dict）
save_path = os.path.join(tempfile.gettempdir(), 'demo_model.pth')

# 保存
torch.save(model.state_dict(), save_path)
print(f"模型已保存到: {save_path}")

# 加载
model_new = DemoNet()  # 先创建模型实例
state_dict = torch.load(save_path, weights_only=True)
model_new.load_state_dict(state_dict)
print("模型已加载")

# 验证参数一致
for (n1, p1), (n2, p2) in zip(
    model.named_parameters(), model_new.named_parameters()
):
    assert torch.equal(p1, p2), f"{n1} 不一致！"
print("验证通过: 所有参数一致")

In [ ]:
# strict 参数: 控制是否严格匹配 key
# strict=True (默认): key 必须完全匹配
# strict=False: 允许缺少或多余的 key

# 模拟部分加载的场景
partial_state = {k: v for k, v in state_dict.items() if 'features' in k}
print(f"只加载 features 部分: {len(partial_state)} 个参数")

model_partial = DemoNet()
result = model_partial.load_state_dict(partial_state, strict=False)
print(f"\nmissing_keys（未加载的）: {result.missing_keys}")
print(f"unexpected_keys（多余的）: {result.unexpected_keys}")

# 清理临时文件
os.remove(save_path)

**`strict=False` 的使用场景：**

1. **迁移学习**：只加载特征提取部分的权重
2. **模型结构变化**：添加/删除了某些层
3. **预训练模型**：key 名称略有不同

## 4. 遍历方法对比

| 方法 | 返回内容 | 递归 |
|------|---------|------|
| `parameters()` | 参数张量 | 是 |
| `named_parameters()` | (名称, 参数) | 是 |
| `modules()` | 所有模块（含自身） | 是（深度优先） |
| `named_modules()` | (名称, 模块) | 是（深度优先） |
| `children()` | 直接子模块 | 否（只一层） |
| `named_children()` | (名称, 子模块) | 否（只一层） |

In [ ]:
model = DemoNet()

# children(): 只返回直接子模块
print("=== children() ===")
for name, child in model.named_children():
    print(f"  {name}: {type(child).__name__}")

In [ ]:
# modules(): 递归返回所有模块（包括自身）
print("=== modules() ===")
for name, module in model.named_modules():
    print(f"  {name or '(self)':30s}  {type(module).__name__}")

**关键区别：**

- `children()` 只看"第一层"子模块（`features` 和 `classifier`）
- `modules()` 递归遍历所有层级，包括 `features.0`（Conv2d）、`features.1`（BatchNorm2d）等

## 5. 精度转换：half(), float(), double(), bfloat16()

In [ ]:
model = DemoNet()

# 默认精度
print(f"默认精度: {next(model.parameters()).dtype}")

# 半精度 (float16)
model_half = DemoNet().half()
print(f"half():    {next(model_half.parameters()).dtype}")

# 双精度 (float64)
model_double = DemoNet().double()
print(f"double():  {next(model_double.parameters()).dtype}")

# bfloat16
model_bf16 = DemoNet().bfloat16()
print(f"bfloat16(): {next(model_bf16.parameters()).dtype}")

# 使用 to() 也可以
model_fp32 = DemoNet().to(torch.float32)
print(f"to(float32): {next(model_fp32.parameters()).dtype}")

**使用场景：**

- `float32`（默认）：训练时的标准精度
- `float16` / `bfloat16`：混合精度训练，加速并减少显存
- `float64`：需要极高精度的科学计算

## 6. apply() - 递归应用函数

`apply(fn)` 会递归地对模型的每个子模块调用 `fn`。最常见的用途是**权重初始化**。

In [ ]:
def init_weights(m):
    """自定义权重初始化。"""
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
        print(f"  初始化 Conv2d: {m}")
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)
        print(f"  初始化 Linear: {m}")
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
        print(f"  初始化 BatchNorm2d: {m}")


model = DemoNet()
print("应用权重初始化:")
model.apply(init_weights)
print("\n初始化完成")

## 7. zero_grad() - 梯度清零

In [ ]:
model = DemoNet()
model.train()

x = torch.randn(2, 1, 28, 28)
output = model(x)
loss = output.sum()
loss.backward()

# 查看梯度
first_param = next(model.parameters())
print(f"反向传播后梯度范数: {first_param.grad.norm().item():.4f}")

# 清零梯度
model.zero_grad()
print(f"zero_grad() 后梯度范数: {first_param.grad.norm().item():.4f}")

print("\n注意: 通常使用 optimizer.zero_grad() 而非 model.zero_grad()")

## 8. 练习题

### 练习 1：完整的模型保存与加载流程

1. 创建模型并做一次前向传播
2. 保存模型参数
3. 创建新模型实例并加载参数
4. 验证两个模型输出一致

In [ ]:
# TODO: 完成模型保存与加载流程

# 1. 创建模型
# model = DemoNet()

# 2. 保存 state_dict
# torch.save(...)

# 3. 加载到新模型
# model_new = DemoNet()
# model_new.load_state_dict(...)

# 4. 验证输出一致
# model.eval()
# model_new.eval()
# x = torch.randn(2, 1, 28, 28)
# with torch.no_grad():
#     out1 = model(x)
#     out2 = model_new(x)
# print(f"输出一致: {torch.equal(out1, out2)}")

### 练习 2：遍历模型并冻结指定层

冻结 `features` 部分的参数，只训练 `classifier` 部分（迁移学习常见操作）。

In [ ]:
# TODO: 冻结 features 的参数
# model = DemoNet()

# for name, param in model.named_parameters():
#     if 'features' in name:
#         # TODO: 冻结该参数
#         pass

# 验证
# for name, param in model.named_parameters():
#     print(f"{name:40s}  requires_grad={param.requires_grad}")

## 9. 小结

### 核心 API

| 方法 | 作用 |
|------|------|
| `train()` / `eval()` | 切换训练/评估模式 |
| `to(device)` / `cpu()` / `cuda()` | 设备转移 |
| `state_dict()` / `load_state_dict()` | 保存/加载参数 |
| `parameters()` / `named_parameters()` | 遍历参数 |
| `modules()` / `children()` | 遍历子模块 |
| `half()` / `float()` / `double()` | 精度转换 |
| `apply(fn)` | 递归应用函数 |
| `zero_grad()` | 梯度清零 |

### 易错点

1. **推理时忘记 `model.eval()`** → BN 和 Dropout 行为不正确
2. **推理时忘记 `torch.no_grad()`** → 浪费内存存储计算图
3. **数据和模型不在同一设备** → RuntimeError
4. **`load_state_dict` 时 key 不匹配** → 需要用 `strict=False`

### 下一步

在下一个 notebook 中，我们将学习 **Hook 机制和 Grad-CAM** 可视化。